# Lab W1c (Breadth): MLP dari Nol dengan Numpy

## Pre-flight Checklist

> [!IMPORTANT]
> Lab ini **breadth opsional** untuk Breadth Check (keluarga MLP). Tidak wajib selesai sebelum W2, tapi sangat dianjurkan saat Anda ingin paham apa yang sebenarnya terjadi di balik `loss.backward()`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W1 sudah dibaca, terutama §2.1 (MLP shape transformer) dan §2.3 (cara kerja backprop).
- Familiar dengan operasi matriks NumPy: `A @ B`, broadcasting, `axis=` argument.
- Ingat chain rule dari §0.5.4 Pendahuluan (rantai turunan = perkalian turunan).
- Lampiran A.1 tersedia jika butuh derivasi 7-langkah formal.

**Yang akan Anda hasilkan di akhir lab:**
- MLP 2-layer (784 → 256 → 10) yang ditulis dari nol di NumPy.
- Forward dan backward pass manual; tidak ada `loss.backward()`.
- Verifikasi finite-difference: gradient analitik vs numerik harus selaras dalam toleransi 1e-3.
- Loss curve sebanding dengan PyTorch autograd (parity check).

**Resource:**
- **Hardware:** CPU cukup. Training 5 epoch MNIST di NumPy ~1-2 menit.
- **Estimasi waktu kerja:** 2-3 jam termasuk derivasi manual dan refleksi.

**Tujuan pedagogis:** mahasiswa menulis forward dan backward pass MLP dua-layer secara manual, lalu melatihnya pada MNIST. Tidak ada PyTorch autograd di sel inti; hanya numpy. Cell terakhir membandingkan dengan `SimpleMLP` PyTorch sebagai parity check.

**Pendamping:** Bab W1 (`01_W1_Tabular_Output_Heads.md`) dan Lampiran A.1 di `14_Lampiran.md` untuk derivasi 7-langkah.

## 1. Setup dan smoke test

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Di Google Colab: clone repo terlebih dahulu, lalu cd ke template_repo/
#   git clone <url-repo> && cd template_repo
# Di environment lokal: jalankan dari folder notebooks/
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Root:", _root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

rng = np.random.default_rng(42)

# Smoke test: muat beberapa batch MNIST agar shape sesuai ekspektasi
tfm = transforms.ToTensor()
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=tfm)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=tfm)

X_train = train_ds.data.numpy().astype(np.float32).reshape(-1, 784) / 255.0
y_train = train_ds.targets.numpy().astype(np.int64)
X_test  = test_ds.data.numpy().astype(np.float32).reshape(-1, 784) / 255.0
y_test  = test_ds.targets.numpy().astype(np.int64)

print('train shape:', X_train.shape, y_train.shape)
print('test shape :', X_test.shape, y_test.shape)
print('pixel range:', X_train.min(), X_train.max())

## 2. Fungsi bangun-balok

Kita akan memakai klasifikasi 10 kelas - softmax + cross-entropy di akhir. Turunan gabungan softmax + CE punya bentuk elegan: `∂L/∂z_out = p - y_onehot` (lihat Section 2.0b untuk turunan singkat). Kita hindari menulis softmax dan CE terpisah karena rentan overflow numerik; gabungan stabil via log-sum-exp trick.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(z.dtype)

def softmax_ce_loss_and_grad(z_out, y_int):
    """Loss rata-rata + gradient ∂L/∂z_out.

    z_out: (B, C) pre-activation output
    y_int: (B,)   integer labels
    """
    B, C = z_out.shape
    z_shift = z_out - z_out.max(axis=1, keepdims=True)   # stabilisasi numerik
    exp = np.exp(z_shift)
    p = exp / exp.sum(axis=1, keepdims=True)
    log_p = z_shift - np.log(exp.sum(axis=1, keepdims=True))
    loss = -log_p[np.arange(B), y_int].mean()
    grad = p.copy()
    grad[np.arange(B), y_int] -= 1.0
    grad /= B
    return loss, grad

# Sanity: gradient harus bernilai kecil ketika prediksi benar sangat percaya diri
z = np.array([[10., 0., 0.]])
L, g = softmax_ce_loss_and_grad(z, np.array([0]))
print('loss low (benar+confident):', round(L, 4))
print('grad small:', g.round(4))

## 3. Inisialisasi parameter (Kaiming untuk ReLU)

In [ ]:
def init_params(d_in=784, d_h=256, d_out=10, seed=42):
    r = np.random.default_rng(seed)
    # Kaiming normal: σ² = 2 / fan_in
    W1 = r.standard_normal((d_h, d_in)).astype(np.float32) * np.sqrt(2.0 / d_in)
    b1 = np.zeros(d_h, dtype=np.float32)
    W2 = r.standard_normal((d_out, d_h)).astype(np.float32) * np.sqrt(2.0 / d_h)
    b2 = np.zeros(d_out, dtype=np.float32)
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

p = init_params()
for k, v in p.items():
    print(k, v.shape, 'std:', v.std().round(4))

## 4. Forward dan backward manual

Ikuti tujuh langkah dari Section 2.0b. Kita simpan aktivasi antara di dict `cache` agar bisa dipakai di backward pass.

In [ ]:
def forward(X, p):
    z1 = X @ p['W1'].T + p['b1']       # (B, d_h)
    h = relu(z1)                        # (B, d_h)
    z2 = h @ p['W2'].T + p['b2']       # (B, d_out)
    cache = {'X': X, 'z1': z1, 'h': h}
    return z2, cache

def backward(grad_z2, cache, p):
    X, z1, h = cache['X'], cache['z1'], cache['h']
    # Langkah 2 dan 3: gradient terhadap W2, b2
    gW2 = grad_z2.T @ h                 # (d_out, d_h)
    gb2 = grad_z2.sum(axis=0)           # (d_out,)
    # Langkah 4: propagasikan ke hidden
    gh = grad_z2 @ p['W2']              # (B, d_h)
    # Langkah 5: lewat turunan ReLU
    gz1 = gh * relu_grad(z1)            # (B, d_h)
    # Langkah 6 dan 7: gradient terhadap W1, b1
    gW1 = gz1.T @ X                     # (d_h, d_in)
    gb1 = gz1.sum(axis=0)               # (d_h,)
    return {'W1': gW1, 'b1': gb1, 'W2': gW2, 'b2': gb2}

## 5. Verifikasi gradient dengan finite differences

Cek: gradient analitik dari `backward` harus mendekati gradient numerik via perturbasi kecil. Kalau gap > 1e-3, ada bug.

> [!NOTE]
> **Kenapa toleransi 1e-3 (bukan 1e-9)?** Finite-difference dengan `eps = 1e-4` menghitung `(L(W+eps) - L(W-eps)) / (2*eps)`. Kesalahan numerik dasar float32 (~1e-7) diperkuat oleh pembagian dengan `eps`, sehingga floor noise berada di sekitar `1e-7 / 1e-4 = 1e-3`. Selisih lebih kecil dari ini sudah masuk noise floating point; selisih lebih besar dari ini biasanya **bug** (salah turunan, salah axis sum, salah transpose). Aturan praktis: 1e-3 adalah threshold pragmatis untuk gradient check di float32; pakai `1e-5` kalau Anda kerja di float64.

In [ ]:
# Ambil 8 sampel saja untuk cek cepat
Xs = X_train[:8]
ys = y_train[:8]

z2, cache = forward(Xs, p)
loss, gz2 = softmax_ce_loss_and_grad(z2, ys)
grads = backward(gz2, cache, p)

# Cek gradient W2 pada 5 posisi acak
eps = 1e-4
W2 = p['W2']
max_diff = 0.0
for _ in range(5):
    i = rng.integers(W2.shape[0])
    j = rng.integers(W2.shape[1])
    orig = W2[i, j]
    W2[i, j] = orig + eps
    L_plus, _ = softmax_ce_loss_and_grad(forward(Xs, p)[0], ys)
    W2[i, j] = orig - eps
    L_minus, _ = softmax_ce_loss_and_grad(forward(Xs, p)[0], ys)
    W2[i, j] = orig
    g_num = (L_plus - L_minus) / (2 * eps)
    g_ana = grads['W2'][i, j]
    diff = abs(g_num - g_ana)
    max_diff = max(max_diff, diff)
    print(f'W2[{i},{j}]  num={g_num:+.6f}  ana={g_ana:+.6f}  |diff|={diff:.2e}')
print(f'\nmax diff: {max_diff:.2e}  (harus < 1e-3)')

## 6. Training loop SGD manual

In [ ]:
def accuracy(X, y, p):
    logits, _ = forward(X, p)
    return (logits.argmax(axis=1) == y).mean()

def train_mlp(X_tr, y_tr, X_va, y_va, epochs=5, batch_size=128, lr=0.1, seed=42):
    p = init_params(seed=seed)
    r = np.random.default_rng(seed)
    n = len(X_tr)
    history = {'train_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        idx = r.permutation(n)
        running = 0.0
        for start in range(0, n, batch_size):
            b = idx[start:start + batch_size]
            Xb, yb = X_tr[b], y_tr[b]
            z2, cache = forward(Xb, p)
            loss, gz2 = softmax_ce_loss_and_grad(z2, yb)
            grads = backward(gz2, cache, p)
            for k in p:
                p[k] -= lr * grads[k]
            running += loss * len(b)
        train_loss = running / n
        val_acc = accuracy(X_va, y_va, p)
        history['train_loss'].append(train_loss)
        history['val_acc'].append(val_acc)
        print(f'epoch {epoch+1:2d}  loss={train_loss:.4f}  val_acc={val_acc:.4f}')
    return p, history

params, hist = train_mlp(X_train, y_train, X_test, y_test, epochs=5, lr=0.1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist['train_loss']); ax[0].set_title('Train loss (numpy MLP)'); ax[0].set_xlabel('epoch')
ax[1].plot(hist['val_acc']);    ax[1].set_title('Val accuracy');           ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()

**Apa yang Anda perhatikan?**

- **Train loss.** Apakah turun monoton, atau ada lonjakan? Lonjakan kecil di tengah epoch wajar (SGD bising); lonjakan tajam = lr terlalu besar.
- **Val accuracy.** Setelah 5 epoch dengan lr=0.1, val_acc target ≈ 0.94-0.96 di MNIST. Kalau Anda dapat di bawah 0.85, kemungkinan ada bug di `backward` (gradient check di §5 lulus tapi mungkin pada subset kecil saja).
- **Bandingkan ke target produksi.** PyTorch `SimpleMLP` di cell §7 akan diuji di lr=0.1 yang sama. Kalau curve numpy Anda **konsisten lebih buruk**, bukan magnitudo (kalau cuma offset 0.01-0.02 itu noise), kemungkinan bug di rumus turunan ReLU atau axis pada `gW1.T @ X`.

---

## 7. Sanity cross-check dengan SimpleMLP PyTorch

Jalankan arsitektur yang identik melalui PyTorch autograd. Loss curve harus menyerupai curve manual di atas (bukan persis sama - seed RNG berbeda, batching berbeda - tapi urutan besaran dan titik konvergensi mirip).

## 8. Refleksi

### 8.1 Pertanyaan Tertarget

1. **Debugging journey.** Di langkah mana backward Anda pertama kali menabrak tembok? Pilih satu yang paling cocok dan tulis 2 kalimat detail:
   - (a) Shape mismatch saat menghitung `gW2.T @ h` atau sejenisnya.
   - (b) Loss menjadi NaN setelah beberapa iterasi.
   - (c) Gradient stuck nol di `gz1` (kemungkinan ReLU mati di seluruh batch).
   - (d) Gradient check `|diff| > 1e-3` walaupun shape benar.
   - (e) Tidak ada masalah - berhasil dari attempt pertama.

2. **Eksperimen sigmoid.** Ganti `relu` di `forward` dengan `sigmoid(z) = 1/(1+exp(-z))`, dan `relu_grad` dengan `sigmoid_grad(z) = sigmoid(z)*(1-sigmoid(z))`. Latih ulang, plot loss curve. Apakah curve turun lebih lambat? Apakah Anda melihat gejala vanishing gradient (gradient di W1 jauh lebih kecil dari W2)? Cetak `np.abs(grads['W1']).mean()` dan `np.abs(grads['W2']).mean()` untuk membandingkan magnitudo.

3. **Threshold gradient check.** Kenapa finite-difference tidak pernah persis sama dengan analytic? Sebutkan dua sumber error numerik (selain yang sudah disebut di NOTE §5). Kapan gap mulai jadi indikator bug, bukan noise?

### 8.2 Self-Check Quick

- [ ] Forward pass shape match: input `(B, 784)` → output `(B, 10)`.
- [ ] Gradient check `max diff < 1e-3` di 5 posisi acak.
- [ ] Training 5 epoch selesai tanpa NaN; val_acc ≥ 0.94.
- [ ] Plot parity numpy vs PyTorch tumpang tindih dalam toleransi 0.05.
- [ ] Saya bisa menjelaskan kenapa `softmax + CE` digabung jadi `p - y_onehot` di gradient.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from src.models import SimpleMLP

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_model = SimpleMLP(input_dim=784, hidden_sizes=(256,), num_classes=10, activation='relu').to(device)
ds_tr = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
dl_tr = DataLoader(ds_tr, batch_size=128, shuffle=True)
opt = optim.SGD(torch_model.parameters(), lr=0.1)
crit = nn.CrossEntropyLoss()
torch_hist = []
for epoch in range(5):
    running = 0.0
    for Xb, yb in dl_tr:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(torch_model(Xb), yb)
        loss.backward()
        opt.step()
        running += loss.item() * len(Xb)
    torch_hist.append(running / len(ds_tr))
    print(f'torch epoch {epoch+1}  loss={torch_hist[-1]:.4f}')

plt.plot(hist['train_loss'], label='numpy manual', marker='o')
plt.plot(torch_hist, label='pytorch autograd', marker='s')
plt.xlabel('epoch'); plt.ylabel('train loss'); plt.legend(); plt.title('Manual vs autograd')
plt.show()

## 8. Refleksi

Jawab tiga pertanyaan di *portofolio mandiri* Anda:

1. Di langkah mana backward Anda pertama kali menabrak tembok? Apa sinyal error yang muncul? (shape mismatch? loss NaN? gradient stuck nol?)
2. Apa yang terjadi pada loss curve jika Anda mengganti `relu` dengan `sigmoid` di `forward` dan `sigmoid_grad` di `backward`? Uji secara langsung. Apakah cocok dengan prediksi *vanishing gradient* di Section 2.0b?
3. Mengapa gradient finite-difference pada cell 5 tidak pernah persis sama dengan gradient analitik? Kapan perbedaan ini mulai jadi indikator bug sungguhan (bukan sekadar noise numerik)?